# HST Corpus 70m Analysis

- Creator: Sebastian Birk (sb75@sanger.ac.uk)
- Date of Creation: 21.02.2025
- Date of Last Modification: 06.04.2025 (by Sebastian Birk; sb75@sanger.ac.uk)

# 1. Setup

## 1.1 Load Libraries 

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("../../utils")

In [3]:
import logging
import os
import warnings
import pickle
import random
from datetime import datetime

import anndata as ad
import torch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scib_metrics
import seaborn as sns
import squidpy as sq
from datasets import load_from_disk
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score
from sklearn.model_selection import train_test_split

from app.infer import infer
from app.train import train
from nichejepa.datasets import prepare_dataset
from nichejepa.utils.config import create_params_from_YAML_wandb_config
from nichejepa.utils.distributed import init_distributed
from nichejepa.utils.evaluation import clustering_metrics, classification_metrics

from analysis_utils import *

/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)


## 1.2 Load Config/Params
All setup will be done in the config file. Please edit that file to continue.

In [4]:
# Set params
dataset = 'hst_corpus_70m'
tokenizer = 'cell_graph'
# load_timestamp = '29032025_080927_875' # gt_tiny
# load_timestamp = '03042025_224852_256' # gt_small
# load_timestamp = '27032025_163938_061' # gt_base
# load_timestamp = '03042025_224013_784' # gt_large
# load_timestamp = None
load_timestamp = '24062025_183131_080'
model_type = 'test'
read_checkpoint = 'nichejepa-ep0_0.pth.tar' # 'nichejepa-ep1_6.pth.tar'

In [5]:
model_config_folder_path = '/nfs/team361/sb75/nichejepa/configs/model' # '/lustre/scratch126/cellgen/team361/sb75/nichejepa/configs/model'
artifacts_folder_path = '../../artifacts'
datasets_folder_path = '/nfs/team361/sb75/DATASETS' # '/lustre/scratch126/cellgen/team361/DATASETS'
token_dict_path = f'{datasets_folder_path}/tokenizer/token_dictionary_homo_sapiens.pkl'

In [6]:
# Set up logger
rank=0
logging.basicConfig()
logger = logging.getLogger()
logger.setLevel(logging.INFO if rank == 0 else logging.ERROR)

In [8]:
# Load config
if not load_timestamp:
    params = create_params_from_YAML_wandb_config(f'{model_config_folder_path}/{dataset}_{model_type}.yaml',logger)
else:
    params = create_params_from_YAML_wandb_config(f'{artifacts_folder_path}/{dataset}/{load_timestamp}/params.yaml',logger)

if read_checkpoint:
    params['state']['read_checkpoint'] = read_checkpoint
if load_timestamp:
    params['state']['load_checkpoint'] = True

params

ERROR:root:YAML configuration file '../../artifacts/hst_corpus_70m/24062025_183131_080/params.yaml' not found.


FileNotFoundError: [Errno 2] No such file or directory: '../../artifacts/hst_corpus_70m/24062025_183131_080/params.yaml'

In [ ]:
# Define folder path for saving and loading artifacts
if not load_timestamp:
    current_timestamp = (
        datetime.now().strftime("%d%m%Y_%H%M%S") +
        f"_{datetime.now().microsecond // 1000:03d}")
    folder_path = os.path.join(artifacts_folder_path,
                               params['data']['dataset_name'],
                               current_timestamp)
else:
    folder_path = f'{artifacts_folder_path}/{dataset}/{load_timestamp}'
    
folder_path

## 1.3 Run Notebook Setup

In [ ]:
os.makedirs(f'{folder_path}/figures', exist_ok=True)

In [ ]:
warnings.filterwarnings("ignore", category=ResourceWarning)

In [ ]:
plt.rcParams['font.size'] = 5
plt.rcParams['text.usetex'] = False
plt.rcParams['svg.fonttype'] = 'none'

In [ ]:
# Improve reproducibility of UMAP and Leiden
os.environ['NUMBA_CPU_NAME'] = 'generic'

In [ ]:
sc.settings.figdir = f'{folder_path}/figures'

sc.settings.set_figure_params(
    dpi=300,
    dpi_save=300,
    figsize=(6, 4),
    facecolor='white',
    fontsize=10
)

## 1.4 Define Functions

In [ ]:
def run_inference(dataset_name,
                  params,
                  read_checkpoint,
                  obs_cols,
                  rerun_inference=False,
                  write_to_disk=True):
    # Load inference results if they exist
    if os.path.exists(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad') and not rerun_inference:
        adata = sc.read_h5ad(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')
    else:
        # Load data
        _, _, test_dataset = prepare_dataset(params)
        test_dataset
        
        # Run model inference
        adata = infer(params,
                      test_dataset,
                      load_folder_path=folder_path,
                      dataset_ids=list(set([batch_id.split('_')[0] for batch_id in params['data']['test_batch_ids']])),
                      obs_cols=obs_cols,
                      agg_type='avg',
                      masked_tokens=None,
                      agg_excluded_tokens=None,
                      cell_gene_ids=[],
                      neighborhood_gene_ids=[])

    # Extract batch from cell ID
    adata.obs["batch"] = [cell_id.split('_')[1].replace('batch', '') for cell_id in adata.obs.index.tolist()]
    
    # Remove missing ground truth
    for obs_col in obs_cols:
        adata = adata[~adata.obs[obs_col].isna()]

    if write_to_disk:
        # Save adata
        adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')

    return adata

In [ ]:
def compute_embedding_similarity(adata,
                                 emb_keys=['cell', 'neighborhood'],
                                 label='niche_type',
                                 write_to_disk=True):
    for emb_key in emb_keys:
        similarity_stats = compute_similarity_stats(
            adata,
            n_samples=None,
            type_key=label,
            embedding_key=f'{emb_key}_emb_layer_{layer}')

        avg_cos_sim = np.mean(similarity_stats["Mean Similarity"])
    
        cosine_df = pd.DataFrame({
            'metric': [f'avg_{label}_cos_sim'],
            'value': [avg_cos_sim]
        })
        print(f'Average {label} cosine similarity is {avg_cos_sim}.')
        cosine_df.to_csv(f'{folder_path}/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_avg_{label}_cos_sim.csv',
                         index=False)
        
        plot_similarity_heatmaps(
            similarity_stats,
            save_file_path=f'{folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{emb_key}_emb_{label}_cos_sim.svg')

In [ ]:
def compute_neighborhood_umap(adata,
                              write_to_disk=True):
    if 'neighborhood_connectivities' not in adata.obsp.keys():
        # Compute UMAP representation for (aggregated) neighborhood gene embeddings
        sc.pp.neighbors(adata,
                        n_neighbors=15,
                        use_rep=f'neighborhood_emb_layer_{layer}',
                        key_added='neighborhood')
    if 'neighborhood_umap' not in adata.obsm.keys():
        sc.tl.umap(adata,
                   neighbors_key='neighborhood',
                   key_added='neighborhood_umap')
    
        if write_to_disk:
            # Save adata
            adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')
    
    if 'cell_connectivities' not in adata.obsp.keys():
        # Compute UMAP representation for (aggregated) cell gene embeddings
        sc.pp.neighbors(adata,
                        n_neighbors=15,
                        use_rep=f'cell_emb_layer_{layer}',
                        key_added='cell')
    if 'cell_umap' not in adata.obsm.keys():
        sc.tl.umap(adata,
                   neighbors_key='cell',
                   key_added='cell_umap')

        if write_to_disk:
            # Save adata
            adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')

    return adata

In [ ]:
def compute_latent_clusters(adata,
                            emb_keys=['cell', 'neighborhood'],
                            obs_col='niche_type',
                            latent_leiden_resolution_cell=1.0,
                            latent_leiden_resolution_neighborhood=0.4,
                            write_to_disk=True):
    for emb_key in emb_keys:
        latent_leiden_resolutions = []
        if emb_key == 'cell':
            latent_leiden_resolution = latent_leiden_resolution_cell
        elif emb_key == 'neighborhood':
            latent_leiden_resolution = latent_leiden_resolution_neighborhood
        adata.obsm['X_umap'] = adata.obsm[f'{emb_key}_umap']
        # Perform clustering until number of ground truth niches is matched
        i = 0
        n_unique_vals = len(adata.obs[obs_col].value_counts() < 100)
        print(f'Target number of unique values: {n_unique_vals}.')
        n_latent_clusters = 0
        while (n_latent_clusters != n_unique_vals):
            if latent_leiden_resolution in latent_leiden_resolutions:
                latent_leiden_resolution += random.uniform(0.0001, 0.001)
            latent_leiden_resolutions.append(latent_leiden_resolution)
            print(f'Leiden resolution: {latent_leiden_resolution}.')
            latent_cluster_key = f'{emb_key}_emb_leiden_res{str(latent_leiden_resolution).replace(".", "_")}'
            i += 1
            sc.tl.leiden(adata,
                         neighbors_key=emb_key,
                         key_added=latent_cluster_key,
                         resolution=latent_leiden_resolution)
        
            n_latent_clusters = len(adata.obs[latent_cluster_key].value_counts() < 100)
            print(f"Number of latent clusters {n_latent_clusters}.")
            if n_latent_clusters == n_unique_vals:
                break
            elif n_latent_clusters < (n_unique_vals - 8):
                latent_leiden_resolution += 0.1
            elif n_latent_clusters < (n_unique_vals - 5):
                latent_leiden_resolution += 0.05
            elif n_latent_clusters < (n_unique_vals - 3):
                latent_leiden_resolution += 0.02
            elif n_latent_clusters < n_unique_vals:
                latent_leiden_resolution += 0.001
            elif n_latent_clusters > (n_unique_vals + 8):
                latent_leiden_resolution -= 0.1
            elif n_latent_clusters > (n_unique_vals + 5):
                latent_leiden_resolution -= 0.05
            elif n_latent_clusters > (n_unique_vals + 3):
                latent_leiden_resolution -= 0.02
            else:
                latent_leiden_resolution -= 0.001
        
        # Compute metrics and store to disk
        labels = adata.obs[obs_col]
        labels_pred = adata.obs[latent_cluster_key]
        nmi = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
        ari = adjusted_rand_score(labels, labels_pred)
        print(f'NMI is {nmi}. ARI is {ari}.')
        metrics_df = pd.DataFrame({
            'metric': ['NMI', 'ARI'],
            'value': [nmi, ari]
        })
        metrics_df.to_csv(f'{folder_path}/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_metrics.csv',
                          index=False)
        
        # Plot UMAPs
        os.makedirs(f'{folder_path}/figures', exist_ok=True)
        labels = ['batch', obs_col, latent_cluster_key]
        # Plot (aggregated) neighborhood gene embeddings, colored by labels
        for label in labels:
            if len(np.unique(adata.obs[label])) > 1:
                sc.pl.umap(adata,
                           neighbors_key=emb_key,
                           title=(emb_key + "_emb_" + label if emb_key not in label else label),
                           color=label,
                           palette=(custom_palette if label == latent_cluster_key else None),
                           show=False)
                plt.savefig(f'{folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}{"_" + emb_key + "_emb" if emb_key not in label else ""}_{label}_umap.svg',
                            dpi=300,
                            bbox_inches='tight')
        
        # Plot niches in tissue
        batch_ids = adata.obs['batch'].unique().tolist()
        for i, batch_id in enumerate(batch_ids):
            adata_batch = adata[adata.obs['batch'] == batch_id]
            sc.pl.spatial(adata_batch,
                          color=latent_cluster_key,
                          palette=custom_palette,
                          spot_size=spot_sizes[i],
                          save=f'{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg')
            os.rename(f'{folder_path}/figures/show{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg',
                      f'{folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg'.replace("show", ""))
    
        if write_to_disk:
            # Save adata
            adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')
    
    return adata

# 2. Analysis

In [ ]:
# Set inference params
layer = params['meta']['enc_depth']
exclude_unshared_tokens = False # True, False
agg_type = 'avg' # 'avg', 'weighted_avg', 'cls'
params['data']['batch_size'] = 128

In [20]:
# Load data
train_dataset, _, test_dataset = prepare_dataset(params)
train_dataset

[Dataset({
     features: ['cell_degrees', 'cell_id', 'batch_token', 'gene_panel_token', 'assay_token', 'species_token', 'tissue_token', 'batch_value_token', 'gene_panel_value_token', 'assay_value_token', 'species_value_token', 'tissue_value_token', 'batch_value', 'gene_panel_value', 'assay_value', 'species_value', 'tissue_value', 'gene_tokens', 'gene_expr', 'seg_tokens', 'n_nonzero_tokens', 'cls_tokens'],
     num_rows: 6942059
 }),
 Dataset({
     features: ['cell_degrees', 'cell_id', 'batch_token', 'gene_panel_token', 'assay_token', 'species_token', 'tissue_token', 'batch_value_token', 'gene_panel_value_token', 'assay_value_token', 'species_value_token', 'tissue_value_token', 'batch_value', 'gene_panel_value', 'assay_value', 'species_value', 'tissue_value', 'gene_tokens', 'gene_expr', 'seg_tokens', 'n_nonzero_tokens', 'cls_tokens'],
     num_rows: 6942059
 }),
 Dataset({
     features: ['cell_degrees', 'cell_id', 'batch_token', 'gene_panel_token', 'assay_token', 'species_token', 'ti

In [21]:
devices=['cuda:0']
world_size=1
os.environ['CUDA_VISIBLE_DEVICES'] = str(devices[rank].split(':')[-1])

logging.basicConfig()
logger = logging.getLogger()
logger.setLevel(logging.INFO if rank == 0 else logging.ERROR)

world_size, rank = init_distributed(rank_and_world_size=(rank, world_size), port=random.randint(40000, 50000))
logger.info(f'Running... (rank: {rank}/{world_size})')

[W605 18:39:36.013460341 Utils.hpp:137] Warning: Environment variable NCCL_ASYNC_ERROR_HANDLING is deprecated; use TORCH_NCCL_ASYNC_ERROR_HANDLING instead (function operator())
INFO:root:Running... (rank: 0/1)


## 2.1 Benchmarking Test Sets

### 2.1.1 xhs49-1b_1p (Xenium Human Skin Melanoma)

#### 2.1.1.1 Define Config

In [ ]:
# Set test batch IDs
params['data']['precomputed_split'] = False
params['data']['test_batch_ids'] = [
    '49_batch0']
params['data']['val_batch_ids'] = None
if 'precomputed_epoch_n_nonzero_tokens' in params['data'].keys():
    del(params['data']['precomputed_epoch_n_nonzero_tokens'])
if 'precomputed_epoch_splits' in params['data'].keys():
    del(params['data']['precomputed_epoch_splits'])

In [ ]:
# Set dataset params
dataset_name = 'xhs49-1b_1p'
niche_key = 'niche_name'
latent_leiden_resolution_cell = 1.0
latent_leiden_resolution_neighborhood = 0.4
spot_sizes = [20] * len(params['data']['test_batch_ids'])

#### 2.1.1.2 Run Model Inference

In [ ]:
adata = run_inference(dataset_name,
                      params,
                      read_checkpoint,
                      obs_cols=[niche_key],
                      write_to_disk=True)

#### 2.1.1.3 Plot Embedding Similarity Stats

In [ ]:
compute_embedding_similarity(adata,
                             emb_keys=['neighborhood'],
                             label=niche_key,
                             write_to_disk=True)

#### 2.1.1.4 Identify Niches & Cell Types

In [ ]:
adata = compute_neighborhood_umap(adata,
                                  write_to_disk=True)

adata = compute_latent_clusters(adata,
                                emb_keys=['neighborhood'],
                                obs_col=niche_key,
                                latent_leiden_resolution_cell=latent_leiden_resolution_cell,
                                latent_leiden_resolution_neighborhood=latent_leiden_resolution_neighborhood,
                                write_to_disk=True)

### 2.1.2 xhs1000-39b_1p (Xenium Human Skin Eczema)

#### 2.1.2.1 Define Config

In [ ]:
# Set test batch IDs
params['data']['precomputed_split'] = False
params['data']['test_batch_ids'] = [
    '1000_batch11',
    '1000_batch19',
    '1000_batch32']
params['data']['val_batch_ids'] = None
if 'precomputed_epoch_n_nonzero_tokens' in params['data'].keys():
    del(params['data']['precomputed_epoch_n_nonzero_tokens'])
if 'precomputed_epoch_splits' in params['data'].keys():
    del(params['data']['precomputed_epoch_splits'])

In [ ]:
# Set dataset params
dataset_name = 'xhs1000-39b_1p'
niche_key = 'niche_type'
batch_key = 'sample'
latent_leiden_resolution_cell = 1.0
latent_leiden_resolution_neighborhood = 0.4
spot_sizes = [20] * len(params['data']['test_batch_ids'])

In [ ]:
#### 2.1.2.2 Run Model Inference

In [ ]:
adata = run_inference(dataset_name,
                      params,
                      read_checkpoint,
                      obs_cols=[niche_key],
                      write_to_disk=True)

In [ ]:
#### 2.1.2.3 Plot Embedding Similarity Stats

In [ ]:
compute_embedding_similarity(adata,
                             emb_keys=['neighborhood'],
                             label=niche_key,
                             write_to_disk=True)

In [ ]:
#### 2.1.2.4 Identify Niches & Cell Types

In [ ]:
adata = compute_neighborhood_umap(adata,
                                  write_to_disk=True)

adata = compute_latent_clusters(adata,
                                emb_keys=['neighborhood'],
                                obs_col=niche_key,
                                latent_leiden_resolution_cell=latent_leiden_resolution_cell,
                                latent_leiden_resolution_neighborhood=latent_leiden_resolution_neighborhood,
                                write_to_disk=True)

### 2.1.3 xhk1007-41b_1p (Xenium Human Kidney RCC)

#### 2.1.3.1 Define Config

In [ ]:
# Set test batch IDs
params['data']['precomputed_split'] = False
params['data']['test_batch_ids'] = [
     '1007_batch1',
     '1007_batch14'
]
params['data']['val_batch_ids'] = None
if 'precomputed_epoch_n_nonzero_tokens' in params['data'].keys():
    del(params['data']['precomputed_epoch_n_nonzero_tokens'])
if 'precomputed_epoch_splits' in params['data'].keys():
    del(params['data']['precomputed_epoch_splits'])

In [ ]:
# Set dataset params
dataset_name = 'xhk1007-41b_1p'
cell_type_key = 'level_1_cell_type'
batch_key = 'batch'
latent_leiden_resolution_cell = 0.75
latent_leiden_resolution_neighborhood = 0.2
spot_sizes = [30] * len(params['data']['test_batch_ids'])

#### 2.1.3.2 Run Model Inference

In [ ]:
adata = run_inference(dataset_name,
                      params,
                      read_checkpoint,
                      obs_cols=[cell_type_key],
                      write_to_disk=True)

#### 2.1.3.3 Plot Embedding Similarity Stats

In [ ]:
compute_embedding_similarity(adata,
                             emb_keys=['cell'],
                             label=cell_type_key,
                             write_to_disk=True)

#### 2.1.3.4 Identify Niches & Cell Types

In [ ]:
adata = compute_neighborhood_umap(adata,
                                  write_to_disk=True)

adata = compute_latent_clusters(adata,
                                emb_keys=['cell'],
                                obs_col=cell_type_key,
                                latent_leiden_resolution_cell=latent_leiden_resolution_cell,
                                latent_leiden_resolution_neighborhood=latent_leiden_resolution_neighborhood,
                                write_to_disk=True)

### 2.1.4 xhb1002-75b_3p (Xenium Human Brain GBM)

#### 2.1.4.1 Define Config

In [ ]:
# Set test batch IDs
params['data']['precomputed_split'] = False
params['data']['test_batch_ids'] = [
    '1002_batch3',
    '1002_batch15'
    '1002_batch27']
params['data']['val_batch_ids'] = None
if 'precomputed_epoch_n_nonzero_tokens' in params['data'].keys():
    del(params['data']['precomputed_epoch_n_nonzero_tokens'])
if 'precomputed_epoch_splits' in params['data'].keys():
    del(params['data']['precomputed_epoch_splits'])

In [ ]:
# Set dataset params
dataset_name = 'xhb1002-75b_3p'
cell_type_key = 'leiden_log1pnorm_res-1'
batch_key = 'batch'
latent_leiden_resolution_cell = 1.0
latent_leiden_resolution_neighborhood = 0.4
spot_sizes = [30] * len(params['data']['test_batch_ids'])

#### 2.1.4.2 Run Model Inference

In [ ]:
adata = run_inference(dataset_name,
                      params,
                      read_checkpoint,
                      obs_cols=[cell_type_key],
                      write_to_disk=True)

#### 2.1.4.3 Plot Embedding Similarity Stats

In [ ]:
compute_embedding_similarity(adata,
                             emb_keys=['cell'],
                             label=cell_type_key,
                             write_to_disk=True)

#### 2.1.4.4 Identify Niches & Cell Types

In [ ]:
adata = compute_neighborhood_umap(adata,
                                  write_to_disk=True)

adata = compute_latent_clusters(adata,
                                emb_keys=['cell'],
                                obs_col=cell_type_key,
                                latent_leiden_resolution_cell=latent_leiden_resolution_cell,
                                latent_leiden_resolution_neighborhood=latent_leiden_resolution_neighborhood,
                                write_to_disk=True)

### 2.1.5 chl59-8b_1p (NanoString CosMx Human Lung NSCLC)

#### 2.1.5.1 Define Config

In [22]:
# Set test batch IDs
params['data']['precomputed_split'] = False
params['data']['test_batch_ids'] = [
    '59_batch0',
    #'59_batch3', # unannotated
    '59_batch6',
    '59_batch7',
    #'59_batch2', # unannotated
    '59_batch4',
    '59_batch1',
    '59_batch5']
params['data']['val_batch_ids'] = None
if 'precomputed_epoch_n_nonzero_tokens' in params['data'].keys():
    del(params['data']['precomputed_epoch_n_nonzero_tokens'])
if 'precomputed_epoch_splits' in params['data'].keys():
    del(params['data']['precomputed_epoch_splits'])

In [23]:
# Set dataset params
dataset_name = 'chl59-8b_1p'
niche_key = 'niche'
batch_key = 'batch'
latent_leiden_resolution_cell = 1.0
latent_leiden_resolution_neighborhood = 0.6
spot_sizes = [80] * len(params['data']['test_batch_ids'])

#### 2.1.5.2 Run Model Inference

In [ ]:
adata = run_inference(dataset_name,
                      params,
                      read_checkpoint,
                      obs_cols=[niche_key],
                      write_to_disk=True)

INFO:root:EncoderMultiMaskWrapper(
  (backbone): GeneTransformerCombinedEncoder(
    (token_embed): Embedding(23407, 384, padding_idx=0)
    (seg_embed): Embedding(12, 384, padding_idx=0)
    (blocks): ModuleList(
      (0-11): 12 x Block(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=384, out_features=1152, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): MLP(
          (fc1): Linear(in_features=384, out_features=1536, bias=True)
          (act): GELU(approximate='none')
          (fc2): Linear(in_features=1536, out_features=384, bias=True)
          (drop): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (norm): LayerNorm((384,), eps=1e-06, elementwise_affi

['encoder', 'predictor', 'target_encoder', 'opt', 'scaler', 'epoch', 'zero_epoch_tracking', 'loss', 'batch_size', 'world_size', 'lr']


0it [00:00, ?it/s]/nfs/team361/sb75/nichejepa/src/app/infer.py:349: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16,
/software/cellgen/team361/sb75/envs/nichejepa/lib/python3.10/contextlib.py:103: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
75it [01:35,  1.25s/it]

#### 2.1.5.3 Plot Embedding Similarity Stats

In [ ]:
compute_embedding_similarity(adata,
                             emb_keys=['neighborhood'],
                             label=niche_key,
                             write_to_disk=True)

#### 2.1.5.4 Identify Niches & Cell Types

In [ ]:
adata = compute_neighborhood_umap(adata,
                                  write_to_disk=True)

adata = compute_latent_clusters(adata,
                                emb_keys=['neighborhood'],
                                obs_col=niche_key,
                                latent_leiden_resolution_cell=latent_leiden_resolution_cell,
                                latent_leiden_resolution_neighborhood=latent_leiden_resolution_neighborhood,
                                write_to_disk=True)

### 2.1.6 chr78-11b_1p (ISS Cartana Human Reproductive Tract)

#### 2.1.6.1 Define Config

In [ ]:
# Set test batch IDs
params['data']['precomputed_split'] = False
params['data']['test_batch_ids'] = [
    '78_batch3',
    '78_batch10',
    '78_batch1',
    '78_batch0',
    '78_batch2',
    '78_batch6',
    '78_batch7',
    '78_batch9',
    '78_batch8',
    '78_batch4',
    '78_batch5']
params['data']['val_batch_ids'] = None
if 'precomputed_epoch_n_nonzero_tokens' in params['data'].keys():
    del(params['data']['precomputed_epoch_n_nonzero_tokens'])
if 'precomputed_epoch_splits' in params['data'].keys():
    del(params['data']['precomputed_epoch_splits'])

In [ ]:
# Set dataset params
dataset_name = 'chr78-11b_1p'
niche_key = 'broad_anatomy'
batch_key = 'batch'
latent_leiden_resolution_cell = 1.0
latent_leiden_resolution_neighborhood = 0.4
spot_sizes = [1920 * 0.093] * len(params['data']['test_batch_ids'])

#### 2.1.6.2 Run Model Inference

In [ ]:
adata = run_inference(dataset_name,
                      params,
                      read_checkpoint,
                      obs_cols=[niche_key],
                      write_to_disk=True)

#### 2.1.6.3 Plot Embedding Similarity Stats

In [ ]:
compute_embedding_similarity(adata,
                             emb_keys=['neighborhood'],
                             label=niche_key,
                             write_to_disk=True)

#### 2.1.6.4 Identify Niches & Cell Types

In [ ]:
adata = compute_neighborhood_umap(adata,
                                  write_to_disk=True)

adata = compute_latent_clusters(adata,
                                emb_keys=['neighborhood'],
                                obs_col=niche_key,
                                latent_leiden_resolution_cell=latent_leiden_resolution_cell,
                                latent_leiden_resolution_neighborhood=latent_leiden_resolution_neighborhood,
                                write_to_disk=True)

## 2.2 Full Corpus

In [ ]:
# Load token dict
with open(token_dict_path, 'rb') as f:
    token_dict = pickle.load(f)

token_dict_reverse =  {value: key for key, value in token_dict.items()}

In [ ]:
# Load dataset
dataset = load_from_disk(params['data']['tokenized_data_folder_path'])

In [ ]:
# Set inference params
layer = params['meta']['enc_depth']
exclude_unshared_tokens = False # True, False
agg_type = 'avg' # 'avg', 'weighted_avg', 'cls'
latent_leiden_resolution = 0.5 # 0.1, 0.5
latent_cluster_key = f'leiden_res{str(latent_leiden_resolution).replace(".", "_")}'

In [ ]:
# Subset dataset
step_size = 2000000
steps = int(np.ceil(len(dataset)/step_size))

dataset_subsets = []

for i in range(steps):
    if ((i+1)*step_size) > len(dataset):
        end_step = len(dataset)
    else:
        end_step = (i+1)*step_size
    dataset_subsets.append(dataset.select(np.arange(i*step_size, end_step)))

del dataset

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i > 5:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[batch_value[0]] for batch_value in dataset_subset['batch_value_token']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i != 6:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[batch_value[0]] for batch_value in dataset_subset['batch_value_token']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 7:
        continue
    if i > 12:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[batch_value[0]] for batch_value in dataset_subset['batch_value_token']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 13:
        continue
    if i > 18:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[batch_value[0]] for batch_value in dataset_subset['batch_value_token']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 19:
        continue
    if i > 25:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[batch_value[0]] for batch_value in dataset_subset['batch_value_token']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i != 26:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[batch_value[0]] for batch_value in dataset_subset['batch_value_token']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 27:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[batch_value[0]] for batch_value in dataset_subset['batch_value_token']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

## 2.3 Analysis Data

### 2.3.1 Joint Skin, Kidney, Embryo

In [ ]:
# Load token dict
with open(token_dict_path, 'rb') as f:
    token_dict = pickle.load(f)

token_dict_reverse =  {value: key for key, value in token_dict.items()}

In [ ]:
# Set test batch IDs
params['data']['precomputed_split'] = False
params['data']['test_batch_ids'] = [
                 '1000_batch0',
                 '1000_batch1',
                 '1000_batch2',
                 '1000_batch3',
                 '1000_batch4',
                 '1000_batch5',
                 '1000_batch6',
                 '1000_batch7',
                 '1000_batch8',
                 '1000_batch9',
                 '1000_batch10',
                 '1000_batch11',
                 '1000_batch12',
                 '1000_batch13',
                 '1000_batch14',
                 '1000_batch15',
                 '1000_batch16',
                 '1000_batch17',
                 '1000_batch18',
                 '1000_batch19',
                 '1000_batch20',
                 '1000_batch21',
                 '1000_batch22',
                 '1000_batch23',
                 '1000_batch24',
                 '1000_batch25',
                 '1000_batch26',
                 '1000_batch27',
                 '1000_batch28',
                 '1000_batch29',
                 '1000_batch30',
                 '1000_batch31',
                 '1000_batch32',
                 '1000_batch33',
                 '1000_batch34',
                 '1000_batch35',
                 '1000_batch36',
                 '1000_batch37',
                 '1000_batch38',
                 '1007_batch0',
                 '1007_batch1',
                 '1007_batch2',
                 '1007_batch3',
                 '1007_batch4',
                 '1007_batch5',
                 '1007_batch6',
                 '1007_batch7',
                 '1007_batch8',
                 '1007_batch9',
                 '1007_batch10',
                 '1007_batch11',
                 '1007_batch12',
                 '1007_batch13',
                 '1007_batch14',
                 '1007_batch15',
                 '1007_batch16',
                 '1007_batch17',
                 '1007_batch18',
                 '1007_batch19',
                 '1007_batch20',
                 '1007_batch21',
                 '1007_batch22',
                 '1007_batch23',
                 '1007_batch24',
                 '1007_batch25',
                 '1007_batch26',
                 '1007_batch27',
                 '1007_batch28',
                 '1007_batch29',
                 '1007_batch30',
                 '1007_batch31',
                 '1007_batch32',
                 '1007_batch33',
                 '1007_batch34',
                 '1007_batch35',
                 '1007_batch36',
                 '1007_batch37',
                 '1007_batch38',
                 '1007_batch39',
                 '1007_batch40',
                 '1008_batch0',
                 '1008_batch1',
                 '1008_batch2',
                 '1008_batch3',
                 '1008_batch4',
                 '1008_batch5',
                 '1008_batch6',
                 '1008_batch7',
                 '1008_batch8',
                 '1008_batch9',
                 '1008_batch10',
                 '1008_batch11',
                 '1008_batch12',
                 '1008_batch13',
                 '1008_batch14',
                 '1008_batch15',
                 '1008_batch16',
                 '1008_batch17',
                 '1008_batch18',
                 '1008_batch19']
params['data']['val_batch_ids'] = None

In [ ]:
# Load data
_, _, dataset = prepare_dataset(params)
dataset

In [ ]:
# Subset dataset
step_size = 2000000
steps = int(np.ceil(len(dataset)/step_size))

dataset_subsets = []

for i in range(steps):
    if ((i+1)*step_size) > len(dataset):
        end_step = len(dataset)
    else:
        end_step = (i+1)*step_size
    dataset_subsets.append(dataset.select(np.arange(i*step_size, end_step)))

del dataset

In [ ]:
# Get batches for current dataset subset
for i, dataset_subset in enumerate(dataset_subsets):
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[batch_value[0]] for batch_value in dataset_subset['batch_value_token']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_1000-1007-1008_{step_size}_{i}.h5ad')
    del adata

### 2.3.2 GBM

In [ ]:
# Load token dict
with open(token_dict_path, 'rb') as f:
    token_dict = pickle.load(f)

token_dict_reverse =  {value: key for key, value in token_dict.items()}

In [ ]:
# Set test batch IDs
params['data']['precomputed_split'] = False
params['data']['test_batch_ids'] = [f'1002_batch{i}' for i in range(75)]
params['data']['val_batch_ids'] = None

In [ ]:
# Load data
_, _, dataset = prepare_dataset(params)
dataset

In [ ]:
# Subset dataset
step_size = 2000000
steps = int(np.ceil(len(dataset)/step_size))

dataset_subsets = []

for i in range(steps):
    if ((i+1)*step_size) > len(dataset):
        end_step = len(dataset)
    else:
        end_step = (i+1)*step_size
    dataset_subsets.append(dataset.select(np.arange(i*step_size, end_step)))

del dataset

In [ ]:
len(dataset_subsets)

In [ ]:
# Get batches for current dataset subset
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 8:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[batch_value[0]] for batch_value in dataset_subset['batch_value_token']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_1002_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Run model inference
adata = infer(params,
              test_dataset,
              load_folder_path=folder_path,
              dataset_ids=list(set([batch_id.split('_')[0] for batch_id in params['data']['test_batch_ids']])),
              agg_type=agg_type,
              masked_tokens=None,
              agg_excluded_tokens=None,
              cell_gene_ids=[],
              neighborhood_gene_ids=[])

# Save adata
# adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')

In [ ]:
# Set inference params
layer = params['meta']['enc_depth']
exclude_unshared_tokens = False # True, False
agg_type = 'avg' # 'avg', 'weighted_avg', 'cls'
latent_leiden_resolution = 0.5 # 0.1, 0.5
latent_cluster_key = f'leiden_res{str(latent_leiden_resolution).replace(".", "_")}'

In [ ]:
item = dataset[0]

In [ ]:
item['cell_id']

In [ ]:
dataset_subset = dataset.filter(lambda x: x['cell_id'].split('_')[0] in batch_ids)

In [ ]:
dataset_subset

In [ ]:
params['data']['tokenized_data_folder_path']

In [ ]:
test_subset = test_dataset.select(range(10))
test_subset
gene_ids = list(np.unique(test_subset['gene_tokens']))

# Run model inference
adata = infer(params,
              test_subset,
              load_folder_path=folder_path,
              dataset_ids=list(set([batch_id.split('_')[0] for batch_id in params['data']['test_batch_ids']])),
              obs_cols=[niche_key],
              agg_type='avg',
              masked_tokens=None,
              agg_excluded_tokens=None,
              cell_gene_ids=[],
              neighborhood_gene_ids=gene_ids)

# Save adata
#adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')

norms = []

for i in range(10):
    for key in adata.obsm.keys():
        if 'neighborhood_emb_gene' in key:
            gene_emb = adata.obsm[key][i]
            norm = np.linalg.norm(gene_emb)
            norms.append(norm)

filtered_data = [x for x in norms if x != 0]

# Plot histogram of filtered data
plt.hist(filtered_data, bins=5, edgecolor='black')